In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv")
test = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv")
stores = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv")
oil = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv")
holidays = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv")
transactions = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Stores:", stores.shape)
print("Oil:", oil.shape)
print("Holidays:", holidays.shape)
print("Transactions:", transactions.shape)

Train: (3000888, 6)
Test: (28512, 5)
Stores: (54, 5)
Oil: (1218, 2)
Holidays: (350, 6)
Transactions: (83488, 3)


In [3]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [4]:
print(train["date"].min())
print(train["date"].max())

print(test["date"].min())
print(test["date"].max())

2013-01-01
2017-08-15
2017-08-16
2017-08-31


In [5]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

In [6]:
for df in [train, test]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

In [7]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [8]:
train = train.merge(
    stores,
    on="store_nbr",
    how="left"
)

test = test.merge(
    stores,
    on="store_nbr",
    how="left"
)

In [9]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [10]:
oil["date"] = pd.to_datetime(oil["date"])

In [11]:
train = train.merge(
    oil,
    on="date",
    how="left"
)

test = test.merge(
    oil,
    on="date",
    how="left"
)

In [12]:
print(train.shape)
display(train.head())

(3000888, 16)


,id,date,store_nbr,family,sales,onpromotion,year,month,day,dayofweek,is_weekend,city,state,type,cluster,dcoilwtico
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,0,Quito,Pichincha,D,13,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1,0,Quito,Pichincha,D,13,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1,0,Quito,Pichincha,D,13,NaN
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1,0,Quito,Pichincha,D,13,NaN
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1,0,Quito,Pichincha,D,13,NaN


In [13]:
print(train.shape)
print(train.dtypes)
print(train.isna().sum().sort_values(ascending=False).head(10))

(3000888, 16)
id                      int64
date           datetime64[ns]
store_nbr               int64
family                 object
sales                 float64
onpromotion             int64
year                    int32
month                   int32
day                     int32
dayofweek               int32
is_weekend              int64
city                   object
state                  object
type                   object
cluster                 int64
dcoilwtico            float64
dtype: object
dcoilwtico     928422
id                  0
store_nbr           0
date                0
sales               0
onpromotion         0
year                0
family              0
month               0
day                 0
dtype: int64


In [14]:
print(train["sales"].describe())

print("\nSkewness:", train["sales"].skew())

count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

Skewness: 7.358757818882655


In [15]:
train["log_sales"] = np.log1p(train["sales"])

print(train["log_sales"].describe())
print("\nLog skewness:", train["log_sales"].skew())

count    3.000888e+06
mean     2.926368e+00
std      2.695122e+00
min      0.000000e+00
25%      0.000000e+00
50%      2.484907e+00
75%      5.282428e+00
max      1.173381e+01
Name: log_sales, dtype: float64

Log skewness: 0.4082954986080495


In [16]:
max_date = train["date"].max()

validation_start = max_date - pd.Timedelta(days=30)

print("Max date:", max_date)
print("Validation starts:", validation_start)

Max date: 2017-08-15 00:00:00
Validation starts: 2017-07-16 00:00:00


In [17]:
train_mask = train["date"] < validation_start
valid_mask = train["date"] >= validation_start

train_part = train.loc[train_mask].copy()
valid_part = train.loc[valid_mask].copy()

print("Train:", train_part.shape)
print("Validation:", valid_part.shape)

print(
    "\nTrain dates:",
    train_part["date"].min(),
    "→",
    train_part["date"].max()
)

print(
    "Validation dates:",
    valid_part["date"].min(),
    "→",
    valid_part["date"].max()
)

Train: (2945646, 17)
Validation: (55242, 17)

Train dates: 2013-01-01 00:00:00 → 2017-07-15 00:00:00
Validation dates: 2017-07-16 00:00:00 → 2017-08-15 00:00:00


In [18]:
group_mean = (
    train_part
    .groupby(["store_nbr", "family"])["sales"]
    .mean()
)

In [19]:
valid_pred = (
    valid_part
    .set_index(["store_nbr", "family"])
    .index
    .map(group_mean)
)

In [20]:
valid_pred = pd.Series(
    valid_pred,
    index=valid_part.index
).fillna(train_part["sales"].mean())

In [21]:
from sklearn.metrics import mean_squared_log_error

baseline_rmsle = np.sqrt(
    mean_squared_log_error(
        valid_part["sales"],
        valid_pred
    )
)

print("Baseline RMSLE:", baseline_rmsle)

Baseline RMSLE: 0.6897067941761713


In [22]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_log_error
import pandas as pd
import numpy as np

feature_cols = [
    "store_nbr",
    "family",
    "year",
    "month",
    "day",
    "dayofweek",
    "is_weekend",
    "onpromotion",
    "city",
    "state",
    "type",
    "cluster",
    "dcoilwtico"
]

cat_cols = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type"
]

X_train = train_part[feature_cols].copy()
X_valid = valid_part[feature_cols].copy()

y_train = train_part["log_sales"]
y_valid = valid_part["sales"]

# Tell the model which columns are categorical
for col in cat_cols:
    categories = X_train[col].astype("category").cat.categories
    
    dtype = pd.api.types.CategoricalDtype(categories=categories)
    
    X_train[col] = X_train[col].astype(dtype)
    X_valid[col] = X_valid[col].astype(dtype)

model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

model.fit(X_train, y_train)

# Model predicts log(1 + sales)
pred_log = model.predict(X_valid)

# Convert back to actual sales
pred = np.expm1(pred_log)

# Sales cannot be negative
pred = np.clip(pred, 0, None)

rmsle = np.sqrt(
    mean_squared_log_error(
        y_valid,
        pred
    )
)

print("Model RMSLE:", rmsle)

Model RMSLE: 0.5271175570470032


In [23]:
train = train.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

test = test.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

group_cols = ["store_nbr", "family"]

for lag in [1, 7, 14, 28]:
    train[f"lag_{lag}"] = (
        train.groupby(group_cols)["sales"]
        .shift(lag)
    )

In [24]:
train[
    ["store_nbr", "family", "date", "sales",
     "lag_1", "lag_7", "lag_14", "lag_28"]
].head(30)

,store_nbr,family,date,sales,lag_1,lag_7,lag_14,lag_28
0,1,AUTOMOTIVE,2013-01-01,0.0,NaN,NaN,NaN,NaN
1,1,AUTOMOTIVE,2013-01-02,2.0,0.0,NaN,NaN,NaN
2,1,AUTOMOTIVE,2013-01-03,3.0,2.0,NaN,NaN,NaN
3,1,AUTOMOTIVE,2013-01-04,3.0,3.0,NaN,NaN,NaN
4,1,AUTOMOTIVE,2013-01-05,5.0,3.0,NaN,NaN,NaN
5,1,AUTOMOTIVE,2013-01-06,2.0,5.0,NaN,NaN,NaN
6,1,AUTOMOTIVE,2013-01-07,0.0,2.0,NaN,NaN,NaN
7,1,AUTOMOTIVE,2013-01-08,2.0,0.0,0.0,NaN,NaN
8,1,AUTOMOTIVE,2013-01-09,2.0,2.0,2.0,NaN,NaN
9,1,AUTOMOTIVE,2013-01-10,2.0,2.0,3.0,NaN,NaN


In [25]:
validation_start = pd.Timestamp("2017-07-16")

train_part = train[
    train["date"] < validation_start
].copy()

valid_part = train[
    train["date"] >= validation_start
].copy()

print(train_part.shape)
print(valid_part.shape)

(2945646, 21)
(55242, 21)


In [26]:
print(
    train_part[["lag_1", "lag_7", "lag_14", "lag_28"]]
    .isna()
    .sum()
)

print(
    valid_part[["lag_1", "lag_7", "lag_14", "lag_28"]]
    .isna()
    .sum()
)

lag_1      1782
lag_7     12474
lag_14    24948
lag_28    49896
dtype: int64
lag_1     0
lag_7     0
lag_14    0
lag_28    0
dtype: int64


In [27]:
# Remove rows where lag features don't exist
train_lag = train_part.dropna(
    subset=["lag_1", "lag_7", "lag_14", "lag_28"]
).copy()

feature_cols_lag = [
    "store_nbr",
    "family",
    "year",
    "month",
    "day",
    "dayofweek",
    "is_weekend",
    "onpromotion",
    "city",
    "state",
    "type",
    "cluster",
    "dcoilwtico",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

cat_cols = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type"
]

X_train = train_lag[feature_cols_lag].copy()
X_valid = valid_part[feature_cols_lag].copy()

y_train = train_lag["log_sales"]
y_valid = valid_part["sales"]

# Keep categorical columns as categorical
for col in cat_cols:
    categories = X_train[col].astype("category").cat.categories
    dtype = pd.api.types.CategoricalDtype(categories=categories)

    X_train[col] = X_train[col].astype(dtype)
    X_valid[col] = X_valid[col].astype(dtype)

model_lag = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

model_lag.fit(X_train, y_train)

pred_log = model_lag.predict(X_valid)

pred = np.expm1(pred_log)
pred = np.clip(pred, 0, None)

rmsle_lag = np.sqrt(
    mean_squared_log_error(
        y_valid,
        pred
    )
)

print("Lag Model RMSLE:", rmsle_lag)

Lag Model RMSLE: 0.40083043486012


In [28]:
for window in [7, 14, 28]:
    train[f"rolling_mean_{window}"] = (
        train
        .groupby(["store_nbr", "family"])["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

In [29]:
for window in [7, 14, 28]:
    train[f"rolling_std_{window}"] = (
        train
        .groupby(["store_nbr", "family"])["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

In [30]:
train_part = train[
    train["date"] < validation_start
].copy()

valid_part = train[
    train["date"] >= validation_start
].copy()

In [31]:
print(
    valid_part[
        [
            "sales",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_28"
        ]
    ].head()
)

      sales  lag_1  lag_7  lag_14  lag_28  rolling_mean_7  rolling_mean_14  \
1653    2.0    6.0    2.0     4.0     2.0        5.428571         4.500000   
1654    2.0    2.0    3.0     0.0     3.0        5.428571         4.357143   
1655    3.0    2.0    7.0     5.0     3.0        5.285714         4.500000   
1656    7.0    3.0    7.0     1.0     1.0        4.714286         4.357143   
1657    4.0    7.0    9.0     5.0     1.0        4.714286         4.785714   

      rolling_mean_28  
1653         4.535714  
1654         4.535714  
1655         4.500000  
1656         4.500000  
1657         4.714286  


In [32]:
rolling_cols = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_28"
]

feature_cols_rolling = feature_cols_lag + rolling_cols

train_rolling = train_part.dropna(
    subset=rolling_cols
).copy()

X_train = train_rolling[feature_cols_rolling].copy()
X_valid = valid_part[feature_cols_rolling].copy()

y_train = train_rolling["log_sales"]
y_valid = valid_part["sales"]

for col in cat_cols:
    categories = X_train[col].astype("category").cat.categories
    dtype = pd.api.types.CategoricalDtype(categories=categories)

    X_train[col] = X_train[col].astype(dtype)
    X_valid[col] = X_valid[col].astype(dtype)

model_rolling = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

model_rolling.fit(X_train, y_train)

pred_log = model_rolling.predict(X_valid)
pred = np.clip(np.expm1(pred_log), 0, None)

rmsle_rolling = np.sqrt(
    mean_squared_log_error(y_valid, pred)
)

print("Rolling Model RMSLE:", rmsle_rolling)

Rolling Model RMSLE: 0.3872147165918052


In [33]:
for lag in [1, 7]:
    train[f"promo_lag_{lag}"] = (
        train
        .groupby(["store_nbr", "family"])["onpromotion"]
        .shift(lag)
    )

for window in [7, 28]:
    train[f"promo_mean_{window}"] = (
        train
        .groupby(["store_nbr", "family"])["onpromotion"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

In [34]:
train_part = train[
    train["date"] < validation_start
].copy()

valid_part = train[
    train["date"] >= validation_start
].copy()

In [35]:
valid_part[
    [
        "date",
        "store_nbr",
        "family",
        "onpromotion",
        "promo_lag_1",
        "promo_lag_7",
        "promo_mean_7",
        "promo_mean_28"
    ]
].head(10)

,date,store_nbr,family,onpromotion,promo_lag_1,promo_lag_7,promo_mean_7,promo_mean_28
1653,2017-07-16,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1654,2017-07-17,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1655,2017-07-18,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1656,2017-07-19,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1657,2017-07-20,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1658,2017-07-21,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1659,2017-07-22,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1660,2017-07-23,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1661,2017-07-24,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0
1662,2017-07-25,1,AUTOMOTIVE,0,0.0,0.0,0.0,0.0


In [36]:
promo_cols = [
    "promo_lag_1",
    "promo_lag_7",
    "promo_mean_7",
    "promo_mean_28"
]

feature_cols_promo = feature_cols_rolling + promo_cols

train_promo = train_part.dropna(
    subset=promo_cols + rolling_cols
).copy()

X_train = train_promo[feature_cols_promo].copy()
X_valid = valid_part[feature_cols_promo].copy()

y_train = train_promo["log_sales"]
y_valid = valid_part["sales"]

for col in cat_cols:
    categories = X_train[col].astype("category").cat.categories
    dtype = pd.api.types.CategoricalDtype(categories=categories)

    X_train[col] = X_train[col].astype(dtype)
    X_valid[col] = X_valid[col].astype(dtype)

model_promo = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

model_promo.fit(X_train, y_train)

pred_log = model_promo.predict(X_valid)
pred = np.clip(np.expm1(pred_log), 0, None)

rmsle_promo = np.sqrt(
    mean_squared_log_error(y_valid, pred)
)

print("Promotion Model RMSLE:", rmsle_promo)

Promotion Model RMSLE: 0.38607100448222603


In [37]:
holidays.head(20)

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
5,2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False
6,2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False


In [38]:
holidays["type"].value_counts()

type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

In [39]:
holidays["locale"].value_counts()

locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

In [40]:
holidays["date"] = pd.to_datetime(holidays["date"])

# Only dates that represent a holiday/event
holiday_dates = holidays[
    holidays["type"].isin(["Holiday", "Event", "Additional", "Bridge"])
]["date"].unique()

workday_dates = holidays[
    holidays["type"] == "Work Day"
]["date"].unique()

train["is_holiday_event"] = (
    train["date"].isin(holiday_dates)
).astype(int)

train["is_work_day"] = (
    train["date"].isin(workday_dates)
).astype(int)

event_dates = holidays[
    holidays["type"] == "Event"
]["date"].unique()

train["is_event"] = (
    train["date"].isin(event_dates)
).astype(int)

In [41]:
train_part = train[
    train["date"] < validation_start
].copy()

valid_part = train[
    train["date"] >= validation_start
].copy()

In [42]:
print(
    train_part[
        ["is_holiday_event", "is_event", "is_work_day"]
    ].head()
)

   is_holiday_event  is_event  is_work_day
0                 1         0            0
1                 0         0            0
2                 0         0            0
3                 0         0            0
4                 0         0            1


In [43]:
train[
    ["date", "is_holiday_event", "is_event", "is_work_day"]
].drop_duplicates().head(20)

,date,is_holiday_event,is_event,is_work_day
0,2013-01-01,1,0,0
1,2013-01-02,0,0,0
2,2013-01-03,0,0,0
3,2013-01-04,0,0,0
4,2013-01-05,0,0,1
5,2013-01-06,0,0,0
6,2013-01-07,0,0,0
7,2013-01-08,0,0,0
8,2013-01-09,0,0,0
9,2013-01-10,0,0,0


In [44]:
calendar_cols = [
    "is_holiday_event",
    "is_event",
    "is_work_day"
]

feature_cols_calendar = feature_cols_promo + calendar_cols

train_calendar = train_part.dropna(
    subset=rolling_cols + promo_cols
).copy()

X_train = train_calendar[feature_cols_calendar].copy()
X_valid = valid_part[feature_cols_calendar].copy()

y_train = train_calendar["log_sales"]
y_valid = valid_part["sales"]

for col in cat_cols:
    categories = X_train[col].astype("category").cat.categories
    dtype = pd.api.types.CategoricalDtype(categories=categories)

    X_train[col] = X_train[col].astype(dtype)
    X_valid[col] = X_valid[col].astype(dtype)

model_calendar = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

model_calendar.fit(X_train, y_train)

pred_log = model_calendar.predict(X_valid)
pred = np.clip(np.expm1(pred_log), 0, None)

rmsle_calendar = np.sqrt(
    mean_squared_log_error(y_valid, pred)
)

print("Calendar Model RMSLE:", rmsle_calendar)

Calendar Model RMSLE: 0.3861157348762849


In [45]:
final_features = (
    feature_cols_lag
    + rolling_cols
    + promo_cols
)

print(len(final_features))
print(final_features)

27
['store_nbr', 'family', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_14', 'rolling_std_28', 'promo_lag_1', 'promo_lag_7', 'promo_mean_7', 'promo_mean_28']


In [46]:
final_train = train.dropna(
    subset=rolling_cols + promo_cols + [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28"
    ]
).copy()

X_final = final_train[final_features].copy()
y_final = final_train["log_sales"]

for col in cat_cols:
    categories = X_final[col].astype("category").cat.categories
    dtype = pd.api.types.CategoricalDtype(categories=categories)

    X_final[col] = X_final[col].astype(dtype)

final_model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

final_model.fit(X_final, y_final)

HistGradientBoostingRegressor(early_stopping=False, l2_regularization=1.0,
                              learning_rate=0.08, max_iter=150,
                              random_state=42)

In [47]:
print(test["date"].min())
print(test["date"].max())
print(test.shape)

2017-08-16 00:00:00
2017-08-31 00:00:00
(28512, 15)


In [48]:
# Keep Kaggle's original order
test = test.sort_values("id").reset_index(drop=True)

print(test[["id", "date"]].head())
print(test[["id", "date"]].tail())

        id       date
0  3000888 2017-08-16
1  3000889 2017-08-16
2  3000890 2017-08-16
3  3000891 2017-08-16
4  3000892 2017-08-16
            id       date
28507  3029395 2017-08-31
28508  3029396 2017-08-31
28509  3029397 2017-08-31
28510  3029398 2017-08-31
28511  3029399 2017-08-31


In [49]:
# Only keep the columns needed for forecasting history
history = train[
    ["store_nbr", "family", "date", "sales"]
].copy()

history = history.sort_values(
    ["store_nbr", "family", "date"]
)

test_dates = sorted(test["date"].unique())

all_predictions = []

for current_date in test_dates:

    print("Predicting:", current_date)

    # Rows for today's prediction
    day_test = test[
        test["date"] == current_date
    ].copy()

    # Combine historical actuals/predictions
    # and calculate lag features
    temp = history.sort_values(
        ["store_nbr", "family", "date"]
    ).copy()

    group = temp.groupby(
        ["store_nbr", "family"]
    )["sales"]

    # Lag features
    for lag in [1, 7, 14, 28]:
        temp[f"lag_{lag}"] = group.shift(lag)

    # Rolling features
    for window in [7, 14, 28]:
        temp[f"rolling_mean_{window}"] = (
            group.transform(
                lambda x: x.shift(1).rolling(window).mean()
            )
        )

        temp[f"rolling_std_{window}"] = (
            group.transform(
                lambda x: x.shift(1).rolling(window).std()
            )
        )

    # Promotion history comes from known promotion values
    # in the original train/test data.
    combined = pd.concat(
        [
            train[
                ["store_nbr", "family", "date", "onpromotion"]
            ],
            test[
                ["store_nbr", "family", "date", "onpromotion"]
            ]
        ],
        ignore_index=True
    )

    combined = combined.sort_values(
        ["store_nbr", "family", "date"]
    )

    promo_group = combined.groupby(
        ["store_nbr", "family"]
    )["onpromotion"]

    for lag in [1, 7]:
        combined[f"promo_lag_{lag}"] = promo_group.shift(lag)

    for window in [7, 28]:
        combined[f"promo_mean_{window}"] = (
            promo_group.transform(
                lambda x: x.shift(1).rolling(window).mean()
            )
        )

    promo_today = combined[
        combined["date"] == current_date
    ].copy()

    # Merge today's test features with calculated history features
    day_features = day_test.merge(
        temp[
            temp["date"] == current_date
        ],
        on=["store_nbr", "family", "date"],
        how="left"
    )

    day_features = day_features.merge(
        promo_today[
            [
                "store_nbr",
                "family",
                "date",
                "promo_lag_1",
                "promo_lag_7",
                "promo_mean_7",
                "promo_mean_28"
            ]
        ],
        on=["store_nbr", "family", "date"],
        how="left"
    )

    # Prepare model input
    X_day = day_features[final_features].copy()

    for col in cat_cols:
        categories = X_final[col].astype("category").cat.categories
        dtype = pd.api.types.CategoricalDtype(categories=categories)
        X_day[col] = X_day[col].astype(dtype)

    # Predict log sales
    pred_log = final_model.predict(X_day)

    # Convert back to sales
    pred_sales = np.clip(
        np.expm1(pred_log),
        0,
        None
    )

    day_features["predicted_sales"] = pred_sales

    all_predictions.append(
        day_features[
            ["id", "store_nbr", "family", "date", "predicted_sales"]
        ]
    )

    # Add predictions to history
    new_history = day_features[
        ["store_nbr", "family", "date", "predicted_sales"]
    ].rename(
        columns={"predicted_sales": "sales"}
    )

    history = pd.concat(
        [history, new_history],
        ignore_index=True
    )

Predicting: 2017-08-16 00:00:00
Predicting: 2017-08-17 00:00:00
Predicting: 2017-08-18 00:00:00
Predicting: 2017-08-19 00:00:00
Predicting: 2017-08-20 00:00:00
Predicting: 2017-08-21 00:00:00
Predicting: 2017-08-22 00:00:00
Predicting: 2017-08-23 00:00:00
Predicting: 2017-08-24 00:00:00
Predicting: 2017-08-25 00:00:00
Predicting: 2017-08-26 00:00:00
Predicting: 2017-08-27 00:00:00
Predicting: 2017-08-28 00:00:00
Predicting: 2017-08-29 00:00:00
Predicting: 2017-08-30 00:00:00
Predicting: 2017-08-31 00:00:00


In [50]:
predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

print(predictions.shape)
print(predictions.head())

(28512, 5)
        id  store_nbr      family       date  predicted_sales
0  3000888          1  AUTOMOTIVE 2017-08-16         0.104750
1  3000889          1   BABY CARE 2017-08-16         0.077740
2  3000890          1      BEAUTY 2017-08-16        20.698827
3  3000891          1   BEVERAGES 2017-08-16        99.892998
4  3000892          1       BOOKS 2017-08-16         0.085118


In [51]:
predictions["predicted_sales"].describe()

count    28512.000000
mean        21.364194
std         30.158430
min          0.000000
25%          0.023746
50%          0.095748
75%         49.237236
max        107.111824
Name: predicted_sales, dtype: float64

In [52]:
print(
    predictions["predicted_sales"].isna().sum()
)

print(
    (predictions["predicted_sales"] < 0).sum()
)

print(
    predictions["predicted_sales"].max()
)

0
0
107.11182359233992


In [53]:
predictions.groupby("date")["predicted_sales"].sum()

date
2017-08-16    50430.469611
2017-08-17    33272.305167
2017-08-18    43506.517150
2017-08-19    40723.331691
2017-08-20    38509.592793
2017-08-21    35421.716894
2017-08-22    35997.802162
2017-08-23    36892.288349
2017-08-24    34441.057647
2017-08-25    41260.577264
2017-08-26    38805.869773
2017-08-27    37276.708365
2017-08-28    33126.503045
2017-08-29    33945.282584
2017-08-30    38388.499564
2017-08-31    37137.374501
Name: predicted_sales, dtype: float64

In [54]:
submission = pd.DataFrame({
    "id": predictions["id"],
    "sales": predictions["predicted_sales"]
})

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())
print(submission.shape)

        id      sales
0  3000888   0.104750
1  3000889   0.077740
2  3000890  20.698827
3  3000891  99.892998
4  3000892   0.085118
(28512, 2)


In [55]:
backtest_start = pd.Timestamp("2017-07-16")
backtest_end = pd.Timestamp("2017-07-31")

backtest = train[
    (train["date"] >= backtest_start) &
    (train["date"] <= backtest_end)
].copy()

history = train[
    train["date"] < backtest_start
][
    ["store_nbr", "family", "date", "sales"]
].copy()

print("History:", history["date"].min(), "→", history["date"].max())
print("Backtest:", backtest["date"].min(), "→", backtest["date"].max())
print("Backtest rows:", len(backtest))

History: 2013-01-01 00:00:00 → 2017-07-15 00:00:00
Backtest: 2017-07-16 00:00:00 → 2017-07-31 00:00:00
Backtest rows: 28512


In [56]:
backtest_predictions = []

# Reset history
history = train[
    train["date"] < backtest_start
][
    ["store_nbr", "family", "date", "sales"]
].copy()

# Static features from the original train data
static_cols = [
    "store_nbr",
    "family",
    "year",
    "month",
    "day",
    "dayofweek",
    "is_weekend",
    "onpromotion",
    "city",
    "state",
    "type",
    "cluster",
    "dcoilwtico"
]

for current_date in sorted(backtest["date"].unique()):

    print("Predicting:", current_date)

    day = backtest[
        backtest["date"] == current_date
    ].copy()

    # ------------------------------------------------
    # SALES HISTORY FEATURES
    # ------------------------------------------------

    temp = history.sort_values(
        ["store_nbr", "family", "date"]
    ).copy()

    group = temp.groupby(
        ["store_nbr", "family"]
    )["sales"]

    for lag in [1, 7, 14, 28]:
        temp[f"lag_{lag}"] = group.shift(lag)

    for window in [7, 14, 28]:

        temp[f"rolling_mean_{window}"] = (
            group.transform(
                lambda x:
                x.shift(1).rolling(window).mean()
            )
        )

        temp[f"rolling_std_{window}"] = (
            group.transform(
                lambda x:
                x.shift(1).rolling(window).std()
            )
        )

    # We need the LAST historical row for each
    # store-family combination.
    latest_features = (
        temp
        .groupby(["store_nbr", "family"])
        .tail(1)
    )

    # ------------------------------------------------
    # PROMOTION HISTORY
    # ------------------------------------------------

    promo_history = train[
        train["date"] <= current_date
    ][
        ["store_nbr", "family", "date", "onpromotion"]
    ].copy()

    promo_history = promo_history.sort_values(
        ["store_nbr", "family", "date"]
    )

    promo_group = promo_history.groupby(
        ["store_nbr", "family"]
    )["onpromotion"]

    for lag in [1, 7]:
        promo_history[f"promo_lag_{lag}"] = (
            promo_group.shift(lag)
        )

    for window in [7, 28]:
        promo_history[f"promo_mean_{window}"] = (
            promo_group.transform(
                lambda x:
                x.shift(1).rolling(window).mean()
            )
        )

    latest_promo = (
        promo_history
        .groupby(["store_nbr", "family"])
        .tail(1)
    )

    # ------------------------------------------------
    # BUILD TODAY'S FEATURES
    # ------------------------------------------------

    day_features = day[static_cols].copy()

    day_features = day_features.merge(
        latest_features[
            [
                "store_nbr",
                "family",
                "lag_1",
                "lag_7",
                "lag_14",
                "lag_28",
                "rolling_mean_7",
                "rolling_mean_14",
                "rolling_mean_28",
                "rolling_std_7",
                "rolling_std_14",
                "rolling_std_28"
            ]
        ],
        on=["store_nbr", "family"],
        how="left"
    )

    day_features = day_features.merge(
        latest_promo[
            [
                "store_nbr",
                "family",
                "promo_lag_1",
                "promo_lag_7",
                "promo_mean_7",
                "promo_mean_28"
            ]
        ],
        on=["store_nbr", "family"],
        how="left"
    )

    # ------------------------------------------------
    # MODEL
    # ------------------------------------------------

    X_day = day_features[final_features].copy()

    for col in cat_cols:
        categories = X_final[col].astype("category").cat.categories
        dtype = pd.api.types.CategoricalDtype(
            categories=categories
        )
        X_day[col] = X_day[col].astype(dtype)

    pred_log = final_model.predict(X_day)

    pred_sales = np.clip(
        np.expm1(pred_log),
        0,
        None
    )

    day_features["predicted_sales"] = pred_sales

    backtest_predictions.append(
        pd.DataFrame({
            "store_nbr": day["store_nbr"].values,
            "family": day["family"].values,
            "date": day["date"].values,
            "sales": day["sales"].values,
            "predicted_sales": pred_sales
        })
    )

    # ------------------------------------------------
    # CRITICAL:
    # ADD PREDICTIONS TO HISTORY
    # ------------------------------------------------

    new_history = pd.DataFrame({
        "store_nbr": day["store_nbr"].values,
        "family": day["family"].values,
        "date": day["date"].values,
        "sales": pred_sales
    })

    history = pd.concat(
        [history, new_history],
        ignore_index=True
    )

Predicting: 2017-07-16 00:00:00
Predicting: 2017-07-17 00:00:00
Predicting: 2017-07-18 00:00:00
Predicting: 2017-07-19 00:00:00
Predicting: 2017-07-20 00:00:00
Predicting: 2017-07-21 00:00:00
Predicting: 2017-07-22 00:00:00
Predicting: 2017-07-23 00:00:00
Predicting: 2017-07-24 00:00:00
Predicting: 2017-07-25 00:00:00
Predicting: 2017-07-26 00:00:00
Predicting: 2017-07-27 00:00:00
Predicting: 2017-07-28 00:00:00
Predicting: 2017-07-29 00:00:00
Predicting: 2017-07-30 00:00:00
Predicting: 2017-07-31 00:00:00


In [57]:
backtest_predictions = pd.concat(
    backtest_predictions,
    ignore_index=True
)

backtest_rmsle = np.sqrt(
    mean_squared_log_error(
        backtest_predictions["sales"],
        backtest_predictions["predicted_sales"]
    )
)

print("Recursive Backtest RMSLE:", backtest_rmsle)

Recursive Backtest RMSLE: 0.4322581819937488


In [58]:
print("Backtest predictions:")
print(
    backtest_predictions["predicted_sales"].describe()
)

print("\nKaggle predictions:")
print(
    predictions["predicted_sales"].describe()
)

Backtest predictions:
count    28512.000000
mean       462.036617
std       1239.166239
min          0.000000
25%          3.185028
50%         26.292799
75%        265.936050
max      12609.116045
Name: predicted_sales, dtype: float64

Kaggle predictions:
count    28512.000000
mean        21.364194
std         30.158430
min          0.000000
25%          0.023746
50%          0.095748
75%         49.237236
max        107.111824
Name: predicted_sales, dtype: float64


In [59]:
print("Backtest daily totals:")
print(
    backtest_predictions
    .groupby("date")["predicted_sales"]
    .sum()
)

print("\nKaggle daily totals:")
print(
    predictions
    .groupby("date")["predicted_sales"]
    .sum()
)

Backtest daily totals:
date
2017-07-16    992726.219721
2017-07-17    909713.439406
2017-07-18    804400.080057
2017-07-19    782223.041004
2017-07-20    741482.535960
2017-07-21    701992.390929
2017-07-22    852933.089601
2017-07-23    934379.589710
2017-07-24    868871.973092
2017-07-25    786684.918345
2017-07-26    739871.621030
2017-07-27    710249.909270
2017-07-28    698475.062918
2017-07-29    815552.139905
2017-07-30    938401.793251
2017-07-31    895630.210843
Name: predicted_sales, dtype: float64

Kaggle daily totals:
date
2017-08-16    50430.469611
2017-08-17    33272.305167
2017-08-18    43506.517150
2017-08-19    40723.331691
2017-08-20    38509.592793
2017-08-21    35421.716894
2017-08-22    35997.802162
2017-08-23    36892.288349
2017-08-24    34441.057647
2017-08-25    41260.577264
2017-08-26    38805.869773
2017-08-27    37276.708365
2017-08-28    33126.503045
2017-08-29    33945.282584
2017-08-30    38388.499564
2017-08-31    37137.374501
Name: predicted_sales, dtyp

In [60]:
print(
    predictions.sort_values(
        "predicted_sales",
        ascending=False
    ).head(20)
)

            id  store_nbr     family       date  predicted_sales
27600  3028488         33  GROCERY I 2017-08-31       107.111824
6735   3007623         48  BEVERAGES 2017-08-19       106.801457
20991  3021879         48  BEVERAGES 2017-08-27       106.774485
19209  3020097         48  BEVERAGES 2017-08-26       106.658428
21000  3021888         48  GROCERY I 2017-08-27       106.496703
27591  3028479         33  BEVERAGES 2017-08-31       106.475136
8517   3009405         48  BEVERAGES 2017-08-20       106.431906
19218  3020106         48  GROCERY I 2017-08-26       106.391473
8526   3009414         48  GROCERY I 2017-08-20       106.001330
20925  3021813         46  BEVERAGES 2017-08-27       105.904971
20892  3021780         45  BEVERAGES 2017-08-27       105.904971
6744   3007632         48  GROCERY I 2017-08-19       105.896585
20958  3021846         47  BEVERAGES 2017-08-27       105.741187
20901  3021789         45  GROCERY I 2017-08-27       105.619610
20859  3021747         44

In [61]:
current_date = pd.Timestamp("2017-08-16")

day = test[
    test["date"] == current_date
].copy()

# Historical sales
temp = history.sort_values(
    ["store_nbr", "family", "date"]
).copy()

group = temp.groupby(
    ["store_nbr", "family"]
)["sales"]

for lag in [1, 7, 14, 28]:
    temp[f"lag_{lag}"] = group.shift(lag)

for window in [7, 14, 28]:
    temp[f"rolling_mean_{window}"] = (
        group.transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    temp[f"rolling_std_{window}"] = (
        group.transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

latest_features = (
    temp
    .groupby(["store_nbr", "family"])
    .tail(1)
)

# Promotion history
promo_history = train[
    train["date"] <= current_date
][
    ["store_nbr", "family", "date", "onpromotion"]
].copy()

promo_history = promo_history.sort_values(
    ["store_nbr", "family", "date"]
)

promo_group = promo_history.groupby(
    ["store_nbr", "family"]
)["onpromotion"]

for lag in [1, 7]:
    promo_history[f"promo_lag_{lag}"] = promo_group.shift(lag)

for window in [7, 28]:
    promo_history[f"promo_mean_{window}"] = (
        promo_group.transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

latest_promo = (
    promo_history
    .groupby(["store_nbr", "family"])
    .tail(1)
)

day_features = day[[
    "store_nbr",
    "family",
    "date",
    "year",
    "month",
    "day",
    "dayofweek",
    "is_weekend",
    "onpromotion",
    "city",
    "state",
    "type",
    "cluster",
    "dcoilwtico"
]].copy()

day_features = day_features.merge(
    latest_features[
        [
            "store_nbr",
            "family",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_28",
            "rolling_std_7",
            "rolling_std_14",
            "rolling_std_28"
        ]
    ],
    on=["store_nbr", "family"],
    how="left"
)

day_features = day_features.merge(
    latest_promo[
        [
            "store_nbr",
            "family",
            "promo_lag_1",
            "promo_lag_7",
            "promo_mean_7",
            "promo_mean_28"
        ]
    ],
    on=["store_nbr", "family"],
    how="left"
)

print(day_features[final_features].describe(include="all").T)

                  count unique         top freq        mean          std  \
store_nbr        1782.0    NaN         NaN  NaN        27.5    15.590159   
family             1782     33  AUTOMOTIVE   54         NaN          NaN   
year             1782.0    NaN         NaN  NaN      2017.0          0.0   
month            1782.0    NaN         NaN  NaN         8.0          0.0   
day              1782.0    NaN         NaN  NaN        16.0          0.0   
dayofweek        1782.0    NaN         NaN  NaN         2.0          0.0   
is_weekend       1782.0    NaN         NaN  NaN         0.0          0.0   
onpromotion      1782.0    NaN         NaN  NaN   17.137486    42.970547   
city               1782     22       Quito  594         NaN          NaN   
state              1782     16   Pichincha  627         NaN          NaN   
type               1782      5           D  594         NaN          NaN   
cluster          1782.0    NaN         NaN  NaN    8.481481     4.651039   
dcoilwtico  

In [62]:
print(
    day_features[
        day_features["store_nbr"] == 1
    ].head(5).T
)

                                   0                    1  \
store_nbr                          1                    1   
family                    AUTOMOTIVE            BABY CARE   
date             2017-08-16 00:00:00  2017-08-16 00:00:00   
year                            2017                 2017   
month                              8                    8   
day                               16                   16   
dayofweek                          2                    2   
is_weekend                         0                    0   
onpromotion                        0                    0   
city                           Quito                Quito   
state                      Pichincha            Pichincha   
type                               D                    D   
cluster                           13                   13   
dcoilwtico                      46.8                 46.8   
lag_1                       2.402419             0.094261   
lag_7                   

In [63]:
print(
    train[
        (train["store_nbr"] == 1) &
        (train["family"] == "AUTOMOTIVE") &
        (train["date"] == "2017-08-15")
    ][
        [
            "date",
            "sales",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_28"
        ]
    ].T
)

                                1683
date             2017-08-15 00:00:00
sales                            4.0
lag_1                            1.0
lag_7                            4.0
lag_14                           5.0
lag_28                           3.0
rolling_mean_7              4.142857
rolling_mean_14             4.785714
rolling_mean_28                  5.0


In [64]:
history_aug15 = train[
    train["date"] <= pd.Timestamp("2017-08-15")
][
    ["store_nbr", "family", "date", "sales"]
].copy()

history_aug15 = history_aug15.sort_values(
    ["store_nbr", "family", "date"]
)

In [65]:
print(
    history_aug15[
        (history_aug15["store_nbr"] == 1) &
        (history_aug15["family"] == "AUTOMOTIVE")
    ].tail(5)
)

      store_nbr      family       date  sales
1679          1  AUTOMOTIVE 2017-08-11    1.0
1680          1  AUTOMOTIVE 2017-08-12    6.0
1681          1  AUTOMOTIVE 2017-08-13    1.0
1682          1  AUTOMOTIVE 2017-08-14    1.0
1683          1  AUTOMOTIVE 2017-08-15    4.0


In [66]:
temp = history_aug15.copy()

group = temp.groupby(
    ["store_nbr", "family"]
)["sales"]

for lag in [1, 7, 14, 28]:
    temp[f"lag_{lag}"] = group.shift(lag)

for window in [7, 14, 28]:
    temp[f"rolling_mean_{window}"] = (
        group.transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    temp[f"rolling_std_{window}"] = (
        group.transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

row = temp[
    (temp["store_nbr"] == 1) &
    (temp["family"] == "AUTOMOTIVE")
].tail(1)

print(
    row[
        [
            "date",
            "sales",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_28"
        ]
    ].T
)

                                1683
date             2017-08-15 00:00:00
sales                            4.0
lag_1                            1.0
lag_7                            4.0
lag_14                           5.0
lag_28                           3.0
rolling_mean_7              4.142857
rolling_mean_14             4.785714
rolling_mean_28                  5.0


In [67]:
latest_features = (
    temp
    .groupby(["store_nbr", "family"])
    .tail(1)
)

day = test[
    test["date"] == pd.Timestamp("2017-08-16")
].copy()

day_features = day[[
    "store_nbr",
    "family",
    "date",
    "year",
    "month",
    "day",
    "dayofweek",
    "is_weekend",
    "onpromotion",
    "city",
    "state",
    "type",
    "cluster",
    "dcoilwtico"
]].copy()

day_features = day_features.merge(
    latest_features[
        [
            "store_nbr",
            "family",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_28",
            "rolling_std_7",
            "rolling_std_14",
            "rolling_std_28"
        ]
    ],
    on=["store_nbr", "family"],
    how="left"
)

In [68]:
train_clean = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv")
test_clean = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv")

train_clean["date"] = pd.to_datetime(train_clean["date"])
test_clean["date"] = pd.to_datetime(test_clean["date"])

train_clean = train_clean.merge(
    stores,
    on="store_nbr",
    how="left"
)

test_clean = test_clean.merge(
    stores,
    on="store_nbr",
    how="left"
)

oil_clean = oil.copy()
oil_clean["date"] = pd.to_datetime(oil_clean["date"])

train_clean = train_clean.merge(
    oil_clean,
    on="date",
    how="left"
)

test_clean = test_clean.merge(
    oil_clean,
    on="date",
    how="left"
)

for df in [train_clean, test_clean]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (
        df["dayofweek"] >= 5
    ).astype(int)

train_clean["log_sales"] = np.log1p(
    train_clean["sales"]
)

In [69]:
train_clean = train_clean.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

group = train_clean.groupby(
    ["store_nbr", "family"]
)["sales"]

for lag in [1, 7, 14, 28]:
    train_clean[f"lag_{lag}"] = group.shift(lag)

for window in [7, 14, 28]:

    train_clean[f"rolling_mean_{window}"] = (
        group.transform(
            lambda x:
            x.shift(1).rolling(window).mean()
        )
    )

    train_clean[f"rolling_std_{window}"] = (
        group.transform(
            lambda x:
            x.shift(1).rolling(window).std()
        )
    )

In [70]:
promo_group = train_clean.groupby(
    ["store_nbr", "family"]
)["onpromotion"]

for lag in [1, 7]:
    train_clean[f"promo_lag_{lag}"] = (
        promo_group.shift(lag)
    )

for window in [7, 28]:
    train_clean[f"promo_mean_{window}"] = (
        promo_group.transform(
            lambda x:
            x.shift(1).rolling(window).mean()
        )
    )

In [71]:
final_features = [
    "store_nbr",
    "family",
    "year",
    "month",
    "day",
    "dayofweek",
    "is_weekend",
    "onpromotion",
    "city",
    "state",
    "type",
    "cluster",
    "dcoilwtico",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_28",
    "promo_lag_1",
    "promo_lag_7",
    "promo_mean_7",
    "promo_mean_28"
]

cat_cols = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type"
]

final_train = train_clean.dropna(
    subset=[
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_14",
        "rolling_std_28",
        "promo_lag_1",
        "promo_lag_7",
        "promo_mean_7",
        "promo_mean_28"
    ]
).copy()

X_final = final_train[final_features].copy()
y_final = final_train["log_sales"]

for col in cat_cols:
    X_final[col] = X_final[col].astype("category")

final_model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

final_model.fit(X_final, y_final)
""
print("Final model trained.")

Final model trained.


In [72]:
# Cell: Recursive test prediction

# Keep only the historical columns needed for dynamic features
history = train_clean[["store_nbr", "family", "date", "sales"]].copy()

# Precompute promotion history using known train + test promotions
promo_all = pd.concat([
    train_clean[["store_nbr", "family", "date", "onpromotion"]],
    test_clean[["store_nbr", "family", "date", "onpromotion"]]
], ignore_index=True)

promo_all = promo_all.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

promo_group = promo_all.groupby(["store_nbr", "family"])["onpromotion"]

for lag in [1, 7]:
    promo_all[f"promo_lag_{lag}"] = promo_group.shift(lag)

for window in [7, 28]:
    promo_all[f"promo_mean_{window}"] = promo_group.transform(
        lambda x: x.shift(1).rolling(window).mean()
    )

# Dates in Kaggle test
test_dates = sorted(test_clean["date"].unique())

predictions = []

for current_date in test_dates:
    
    # Rows for this prediction day
    day = test_clean[test_clean["date"] == current_date].copy()
    
    # -----------------------------
    # Dynamic sales features
    # -----------------------------
    temp = history.sort_values(
        ["store_nbr", "family", "date"]
    ).copy()

    sales_group = temp.groupby(["store_nbr", "family"])["sales"]

    for lag in [1, 7, 14, 28]:
        temp[f"lag_{lag}"] = sales_group.shift(lag)

    for window in [7, 14, 28]:
        temp[f"rolling_mean_{window}"] = sales_group.transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
        
        temp[f"rolling_std_{window}"] = sales_group.transform(
            lambda x: x.shift(1).rolling(window).std()
        )

    # Latest row available before current date
    dynamic_cols = [
        "store_nbr", "family",
        "lag_1", "lag_7", "lag_14", "lag_28",
        "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
        "rolling_std_7", "rolling_std_14", "rolling_std_28"
    ]

    latest = (
        temp[temp["date"] < current_date]
        .sort_values("date")
        .groupby(["store_nbr", "family"])
        .tail(1)
    )

    latest = latest[dynamic_cols]

    day = day.merge(
        latest,
        on=["store_nbr", "family"],
        how="left"
    )

    # -----------------------------
    # Promotion features
    # -----------------------------
    promo_day = promo_all[
        promo_all["date"] == current_date
    ][
        [
            "store_nbr", "family",
            "promo_lag_1", "promo_lag_7",
            "promo_mean_7", "promo_mean_28"
        ]
    ]

    day = day.merge(
        promo_day,
        on=["store_nbr", "family"],
        how="left"
    )

    # -----------------------------
    # Prepare model input
    # -----------------------------
    X_day = day[final_features].copy()

    for col in cat_cols:
        X_day[col] = X_day[col].astype("category")

    # Match training categorical categories
    for col in cat_cols:
        X_day[col] = X_day[col].cat.set_categories(
            X_final[col].cat.categories
        )

    # Safety check
    if X_day[final_features].isna().any().any():
        print(
            current_date,
            "NaNs:",
            X_day[final_features].isna().sum().sum()
        )

    # -----------------------------
    # Predict
    # -----------------------------
    pred_log = final_model.predict(X_day)

    pred_sales = np.maximum(
        np.expm1(pred_log),
        0
    )

    day["pred_sales"] = pred_sales

    predictions.append(
        day[["id", "store_nbr", "family", "date", "pred_sales"]]
    )

    # Feed predictions back into history
    history = pd.concat([
        history,
        day[["store_nbr", "family", "date"]].assign(
            sales=pred_sales
        )
    ], ignore_index=True)

    print(
        current_date.date(),
        "mean:",
        round(pred_sales.mean(), 2),
        "total:",
        round(pred_sales.sum(), 2)
    )

pred_df = pd.concat(predictions, ignore_index=True)

print("\nPrediction shape:", pred_df.shape)
print(pred_df["pred_sales"].describe())

2017-08-16 mean: 436.06 total: 777067.49
2017-08-17 mean: 402.47 total: 717198.6
2017-08-18 mean: 395.17 total: 704193.04
2017-08-19 00:00:00 NaNs: 1782
2017-08-19 mean: 471.42 total: 840075.76
2017-08-20 00:00:00 NaNs: 1782
2017-08-20 mean: 488.76 total: 870974.97
2017-08-21 mean: 451.79 total: 805097.09
2017-08-22 mean: 419.16 total: 746937.86
2017-08-23 mean: 403.88 total: 719717.68
2017-08-24 mean: 386.52 total: 688769.94
2017-08-25 mean: 383.09 total: 682663.39
2017-08-26 00:00:00 NaNs: 1782
2017-08-26 mean: 454.43 total: 809796.14
2017-08-27 00:00:00 NaNs: 1782
2017-08-27 mean: 471.62 total: 840422.86
2017-08-28 mean: 433.9 total: 773217.98
2017-08-29 mean: 416.78 total: 742706.74
2017-08-30 mean: 430.89 total: 767839.12
2017-08-31 mean: 426.16 total: 759416.23

Prediction shape: (28512, 5)
count    28512.000000
mean       429.506695
std       1134.189279
min          0.000000
25%          4.089267
50%         26.981257
75%        257.397405
max      11448.296732
Name: pred_sales

In [73]:
pred_df.sort_values("id").head()
pred_df.sort_values("id").tail()

,id,store_nbr,family,date,pred_sales
28507,3029395,9,POULTRY,2017-08-31,338.355208
28508,3029396,9,PREPARED FOODS,2017-08-31,99.530162
28509,3029397,9,PRODUCE,2017-08-31,1599.724898
28510,3029398,9,SCHOOL AND OFFICE SUPPLIES,2017-08-31,153.230884
28511,3029399,9,SEAFOOD,2017-08-31,13.248169


In [74]:
pred_df.groupby("date")["pred_sales"].sum()

date
2017-08-16    777067.487367
2017-08-17    717198.600688
2017-08-18    704193.040418
2017-08-19    840075.759424
2017-08-20    870974.972293
2017-08-21    805097.094401
2017-08-22    746937.855335
2017-08-23    719717.679683
2017-08-24    688769.938553
2017-08-25    682663.389409
2017-08-26    809796.140089
2017-08-27    840422.855068
2017-08-28    773217.984345
2017-08-29    742706.741603
2017-08-30    767839.118043
2017-08-31    759416.230258
Name: pred_sales, dtype: float64

In [75]:
# Cell: Create submission

submission = (
    pred_df[["id", "pred_sales"]]
    .rename(columns={"pred_sales": "sales"})
    .sort_values("id")
)

submission.to_csv("submission.csv", index=False)

print(submission.shape)
print(submission.head())
print(submission.tail())

(28512, 2)
        id        sales
0  3000888     3.367974
1  3000889     0.040488
2  3000890     4.642478
3  3000891  2177.306208
4  3000892     0.042672
            id        sales
28507  3029395   338.355208
28508  3029396    99.530162
28509  3029397  1599.724898
28510  3029398   153.230884
28511  3029399    13.248169


In [76]:
# Find exactly which features are NaN on Aug 19

check_date = pd.Timestamp("2017-08-19")

day = test_clean[test_clean["date"] == check_date].copy()

# Rebuild history exactly as it stands before Aug 19
temp = history.sort_values(
    ["store_nbr", "family", "date"]
).copy()

sales_group = temp.groupby(["store_nbr", "family"])["sales"]

for lag in [1, 7, 14, 28]:
    temp[f"lag_{lag}"] = sales_group.shift(lag)

for window in [7, 14, 28]:
    temp[f"rolling_mean_{window}"] = sales_group.transform(
        lambda x: x.shift(1).rolling(window).mean()
    )
    temp[f"rolling_std_{window}"] = sales_group.transform(
        lambda x: x.shift(1).rolling(window).std()
    )

latest = (
    temp[temp["date"] < check_date]
    .sort_values("date")
    .groupby(["store_nbr", "family"])
    .tail(1)
)

latest = latest[
    [
        "store_nbr", "family",
        "lag_1", "lag_7", "lag_14", "lag_28",
        "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
        "rolling_std_7", "rolling_std_14", "rolling_std_28"
    ]
]

day = day.merge(
    latest,
    on=["store_nbr", "family"],
    how="left"
)

promo_day = promo_all[
    promo_all["date"] == check_date
][
    [
        "store_nbr", "family",
        "promo_lag_1", "promo_lag_7",
        "promo_mean_7", "promo_mean_28"
    ]
]

day = day.merge(
    promo_day,
    on=["store_nbr", "family"],
    how="left"
)

X_check = day[final_features]

print("NaNs by feature:")
print(X_check.isna().sum()[X_check.isna().sum() > 0])

print("\nNaN percentage:")
print(
    (X_check.isna().mean() * 100)
    .sort_values(ascending=False)
    .head(10)
)

NaNs by feature:
dcoilwtico    1782
dtype: int64

NaN percentage:
dcoilwtico     100.0
family           0.0
store_nbr        0.0
month            0.0
day              0.0
dayofweek        0.0
is_weekend       0.0
onpromotion      0.0
city             0.0
state            0.0
dtype: float64


In [77]:
# Build a complete daily oil-price series

oil_filled = oil.copy()
oil_filled["date"] = pd.to_datetime(oil_filled["date"])

# Complete calendar covering the entire competition period
all_dates = pd.date_range(
    start=min(train_clean["date"].min(), test_clean["date"].min()),
    end=max(train_clean["date"].max(), test_clean["date"].max()),
    freq="D"
)

oil_filled = (
    oil_filled
    .set_index("date")
    .reindex(all_dates)
    .rename_axis("date")
    .reset_index()
)

# Fill oil prices across missing dates
oil_filled["dcoilwtico"] = (
    oil_filled["dcoilwtico"]
    .ffill()
    .bfill()
)

print(oil_filled.head())
print(oil_filled.tail())

print(
    "Oil NaNs:",
    oil_filled["dcoilwtico"].isna().sum()
)

        date  dcoilwtico
0 2013-01-01       93.14
1 2013-01-02       93.14
2 2013-01-03       92.97
3 2013-01-04       93.12
4 2013-01-05       93.12
           date  dcoilwtico
1699 2017-08-27       47.65
1700 2017-08-28       46.40
1701 2017-08-29       46.46
1702 2017-08-30       45.96
1703 2017-08-31       47.26
Oil NaNs: 0


In [78]:
train_clean = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv")
test_clean = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv")

train_clean["date"] = pd.to_datetime(train_clean["date"])
test_clean["date"] = pd.to_datetime(test_clean["date"])

# Store features
train_clean = train_clean.merge(
    stores,
    on="store_nbr",
    how="left"
)

test_clean = test_clean.merge(
    stores,
    on="store_nbr",
    how="left"
)

# Complete oil series
train_clean = train_clean.merge(
    oil_filled,
    on="date",
    how="left"
)

test_clean = test_clean.merge(
    oil_filled,
    on="date",
    how="left"
)

# Calendar features
for df in [train_clean, test_clean]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

print("Train oil NaNs:", train_clean["dcoilwtico"].isna().sum())
print("Test oil NaNs:", test_clean["dcoilwtico"].isna().sum())

Train oil NaNs: 0
Test oil NaNs: 0


In [79]:
# CLEAN RESET: reload everything from raw files

import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor

# -----------------------------
# 1. Reload raw data
# -----------------------------

train_clean = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv")
test_clean = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv")
stores_clean = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv")
oil_clean = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv")

train_clean["date"] = pd.to_datetime(train_clean["date"])
test_clean["date"] = pd.to_datetime(test_clean["date"])
stores_clean = stores_clean.copy()
oil_clean["date"] = pd.to_datetime(oil_clean["date"])

print("Raw train:", train_clean.shape)
print("Raw test:", test_clean.shape)

# -----------------------------
# 2. Build complete oil series
# -----------------------------

all_dates = pd.date_range(
    start=train_clean["date"].min(),
    end=test_clean["date"].max(),
    freq="D"
)

oil_daily = (
    oil_clean
    .groupby("date", as_index=True)["dcoilwtico"]
    .mean()
    .reindex(all_dates)
    .rename_axis("date")
    .reset_index()
)

oil_daily["dcoilwtico"] = (
    oil_daily["dcoilwtico"]
    .ffill()
    .bfill()
)

print("Oil NaNs:", oil_daily["dcoilwtico"].isna().sum())

# -----------------------------
# 3. Merge stores + oil
# -----------------------------

train_clean = train_clean.merge(
    stores_clean,
    on="store_nbr",
    how="left"
)

test_clean = test_clean.merge(
    stores_clean,
    on="store_nbr",
    how="left"
)

train_clean = train_clean.merge(
    oil_daily,
    on="date",
    how="left"
)

test_clean = test_clean.merge(
    oil_daily,
    on="date",
    how="left"
)

# -----------------------------
# 4. Calendar features
# -----------------------------

for df in [train_clean, test_clean]:

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (
        df["dayofweek"] >= 5
    ).astype(int)

# -----------------------------
# 5. Verify everything
# -----------------------------

print("\nTrain columns:")
print(train_clean.columns.tolist())

print("\nTest columns:")
print(test_clean.columns.tolist())

print("\nTrain oil NaNs:",
      train_clean["dcoilwtico"].isna().sum())

print("Test oil NaNs:",
      test_clean["dcoilwtico"].isna().sum())

print("\nSales exists:",
      "sales" in train_clean.columns)

print("\nTrain shape:", train_clean.shape)
print("Test shape:", test_clean.shape)

Raw train: (3000888, 6)
Raw test: (28512, 5)
Oil NaNs: 0

Train columns:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'year', 'month', 'day', 'dayofweek', 'is_weekend']

Test columns:
['id', 'date', 'store_nbr', 'family', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'year', 'month', 'day', 'dayofweek', 'is_weekend']

Train oil NaNs: 0
Test oil NaNs: 0

Sales exists: True

Train shape: (3000888, 16)
Test shape: (28512, 15)


In [80]:
# Feature engineering on clean train

train_clean["log_sales"] = np.log1p(train_clean["sales"])

train_clean = train_clean.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

sales_group = train_clean.groupby(
    ["store_nbr", "family"]
)["sales"]

# Sales lags
for lag in [1, 7, 14, 28]:
    train_clean[f"lag_{lag}"] = sales_group.shift(lag)

# Rolling sales statistics
for window in [7, 14, 28]:
    train_clean[f"rolling_mean_{window}"] = sales_group.transform(
        lambda x: x.shift(1).rolling(window).mean()
    )
    
    train_clean[f"rolling_std_{window}"] = sales_group.transform(
        lambda x: x.shift(1).rolling(window).std()
    )

# Promotion history
promo_group = train_clean.groupby(
    ["store_nbr", "family"]
)["onpromotion"]

for lag in [1, 7]:
    train_clean[f"promo_lag_{lag}"] = promo_group.shift(lag)

for window in [7, 28]:
    train_clean[f"promo_mean_{window}"] = promo_group.transform(
        lambda x: x.shift(1).rolling(window).mean()
    )

dynamic_cols = [
    "lag_1", "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
    "rolling_std_7", "rolling_std_14", "rolling_std_28",
    "promo_lag_1", "promo_lag_7",
    "promo_mean_7", "promo_mean_28"
]

print("Dynamic feature NaNs:")
print(train_clean[dynamic_cols].isna().sum())

Dynamic feature NaNs:
lag_1               1782
lag_7              12474
lag_14             24948
lag_28             49896
rolling_mean_7     12474
rolling_mean_14    24948
rolling_mean_28    49896
rolling_std_7      12474
rolling_std_14     24948
rolling_std_28     49896
promo_lag_1         1782
promo_lag_7        12474
promo_mean_7       12474
promo_mean_28      49896
dtype: int64


In [81]:
final_features = [
    "store_nbr", "family",
    "year", "month", "day", "dayofweek", "is_weekend",
    "onpromotion",
    "city", "state", "type", "cluster", "dcoilwtico",
    "lag_1", "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
    "rolling_std_7", "rolling_std_14", "rolling_std_28",
    "promo_lag_1", "promo_lag_7",
    "promo_mean_7", "promo_mean_28"
]

cat_cols = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type"
]

final_train = train_clean.dropna(
    subset=dynamic_cols
).copy()

X_final = final_train[final_features].copy()
y_final = final_train["log_sales"]

for col in cat_cols:
    X_final[col] = X_final[col].astype("category")

final_model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

final_model.fit(X_final, y_final)

print("Final model trained.")
print("Training rows:", len(final_train))

Final model trained.
Training rows: 2950992


In [82]:
promo_all = pd.concat([
    train_clean[
        ["store_nbr", "family", "date", "onpromotion"]
    ],
    test_clean[
        ["store_nbr", "family", "date", "onpromotion"]
    ]
], ignore_index=True)

promo_all = promo_all.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

promo_group = promo_all.groupby(
    ["store_nbr", "family"]
)["onpromotion"]

for lag in [1, 7]:
    promo_all[f"promo_lag_{lag}"] = promo_group.shift(lag)

for window in [7, 28]:
    promo_all[f"promo_mean_{window}"] = promo_group.transform(
        lambda x: x.shift(1).rolling(window).mean()
    )

print(
    "Promotion feature NaNs:",
    promo_all[
        ["promo_lag_1", "promo_lag_7",
         "promo_mean_7", "promo_mean_28"]
    ].isna().sum()
)

Promotion feature NaNs: promo_lag_1       1782
promo_lag_7      12474
promo_mean_7     12474
promo_mean_28    49896
dtype: int64


In [83]:
history = train_clean[
    ["store_nbr", "family", "date", "sales"]
].copy()

test_dates = sorted(test_clean["date"].unique())

predictions = []

for current_date in test_dates:

    day = test_clean[
        test_clean["date"] == current_date
    ].copy()

    # -----------------------------
    # Sales lag / rolling features
    # -----------------------------
    temp = history.sort_values(
        ["store_nbr", "family", "date"]
    )

    sales_group = temp.groupby(
        ["store_nbr", "family"]
    )["sales"]

    for lag in [1, 7, 14, 28]:
        temp[f"lag_{lag}"] = sales_group.shift(lag)

    for window in [7, 14, 28]:

        temp[f"rolling_mean_{window}"] = sales_group.transform(
            lambda x: x.shift(1).rolling(window).mean()
        )

        temp[f"rolling_std_{window}"] = sales_group.transform(
            lambda x: x.shift(1).rolling(window).std()
        )

    dynamic = temp[
        temp["date"] < current_date
    ].sort_values("date").groupby(
        ["store_nbr", "family"]
    ).tail(1)

    dynamic = dynamic[
        [
            "store_nbr", "family",
            "lag_1", "lag_7", "lag_14", "lag_28",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_28",
            "rolling_std_7",
            "rolling_std_14",
            "rolling_std_28"
        ]
    ]

    day = day.merge(
        dynamic,
        on=["store_nbr", "family"],
        how="left"
    )

    # -----------------------------
    # Promotion features
    # -----------------------------
    promo_day = promo_all[
        promo_all["date"] == current_date
    ][
        [
            "store_nbr", "family",
            "promo_lag_1", "promo_lag_7",
            "promo_mean_7", "promo_mean_28"
        ]
    ]

    day = day.merge(
        promo_day,
        on=["store_nbr", "family"],
        how="left"
    )

    # -----------------------------
    # Model input
    # -----------------------------
    X_day = day[final_features].copy()

    for col in cat_cols:
        X_day[col] = X_day[col].astype("category")
        X_day[col] = X_day[col].cat.set_categories(
            X_final[col].cat.categories
        )

    # Check for unexpected NaNs
    nan_count = X_day[final_features].isna().sum().sum()

    if nan_count > 0:
        print(
            current_date,
            "NaNs:",
            nan_count,
            X_day[final_features]
            .isna()
            .sum()
            .loc[lambda x: x > 0]
            .to_dict()
        )

    # -----------------------------
    # Predict
    # -----------------------------
    pred_log = final_model.predict(X_day)

    pred_sales = np.maximum(
        np.expm1(pred_log),
        0
    )

    day["pred_sales"] = pred_sales

    predictions.append(
        day[
            ["id", "store_nbr", "family", "date", "pred_sales"]
        ]
    )

    # Add predictions to history
    history = pd.concat([
        history,
        day[
            ["store_nbr", "family", "date"]
        ].assign(
            sales=pred_sales
        )
    ], ignore_index=True)

    print(
        current_date.date(),
        "mean:", round(pred_sales.mean(), 2),
        "total:", round(pred_sales.sum(), 2)
    )

pred_df = pd.concat(
    predictions,
    ignore_index=True
)

print("\nFinished.")
print("Shape:", pred_df.shape)
print(pred_df["pred_sales"].describe())

2017-08-16 mean: 435.97 total: 776898.4
2017-08-17 mean: 402.04 total: 716442.22
2017-08-18 mean: 397.73 total: 708752.68
2017-08-19 mean: 470.59 total: 838599.77
2017-08-20 mean: 488.18 total: 869929.68
2017-08-21 mean: 453.44 total: 808035.19
2017-08-22 mean: 418.2 total: 745233.32
2017-08-23 mean: 404.83 total: 721398.62
2017-08-24 mean: 386.08 total: 687996.34
2017-08-25 mean: 385.74 total: 687381.47
2017-08-26 mean: 455.31 total: 811363.14
2017-08-27 mean: 473.88 total: 844459.23
2017-08-28 mean: 436.9 total: 778561.17
2017-08-29 mean: 417.31 total: 743643.45
2017-08-30 mean: 434.56 total: 774385.88
2017-08-31 mean: 429.44 total: 765256.59

Finished.
Shape: (28512, 5)
count    28512.000000
mean       430.637526
std       1139.600122
min          0.000000
25%          4.122791
50%         27.946652
75%        258.022319
max      11703.699331
Name: pred_sales, dtype: float64


In [84]:
submission = (
    pred_df[
        ["id", "pred_sales"]
    ]
    .rename(columns={"pred_sales": "sales"})
    .sort_values("id")
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("Submission created.")
print("Shape:", submission.shape)
print("Rows:", len(submission))
print("Unique IDs:", submission["id"].nunique())
print("Missing:", submission["sales"].isna().sum())
print("Negative:", (submission["sales"] < 0).sum())

submission.head()

Submission created.
Shape: (28512, 2)
Rows: 28512
Unique IDs: 28512
Missing: 0
Negative: 0


,id,sales
0,3000888,3.356825
1,3000889,0.033166
2,3000890,4.382565
3,3000891,2236.099880
4,3000892,0.047106
